# Hands-on RAG for Education
## Notebook 2: Build a RAG assistant from scratch

ACM SIGCITE 2026 tutorial · Yong Zheng, Illinois Institute of Technology

**Lab version** for the tutorial JupyterHub.

In Notebook 1 you tried three ready-made tools. Here you build the fourth approach yourself, in five steps, on the same two documents and the same five questions:

| Step | Simple way | Better way |
| --- | --- | --- |
| 1. Convert documents | Extract plain text | Markdown that keeps headings and tables, plus preprocessing |
| 2. Chunk | Fixed-length pieces | Pieces that follow the headings and carry their section path |
| 3. Embed and retrieve | Query without the model's prefix | Query with the prefix; choose how many passages to keep |
| 4. Generate | A plain prompt | A prompt that requires citations and refusal |
| 5. Evaluate | | Score Q1–Q5 and compare all four approaches |

In each step you will **guess** which way works better, **run** both, and **change one setting** yourself. No programming is needed: you change a number or a sentence. Boxes marked **Core idea** point out the few lines of code worth remembering.

The code in this notebook is the code that explains RAG. Everything else (downloads, the connection to the tutorial server, the self-checks, and the score charts) is in `edrag_utils.py`, which you can open from the file browser if you are curious.

**How to run:** click a cell and press **Shift + Enter**. Run the cells from top to bottom.

## 0. Setup

This cell loads the helper file, downloads the documents, connects to the tutorial server, and finds the scores you saved in Notebook 1.

In [ ]:
import re, html, pathlib, urllib.request, importlib
import numpy as np
import pandas as pd
from bs4 import BeautifulSoup
from pypdf import PdfReader
from html_to_markdown import convert
from openai import OpenAI
from IPython.display import Markdown, display

# Helper code shared by both notebooks: downloads, guesses, self-checks, and score charts
try:
    urllib.request.urlretrieve("https://academictutorials.github.io/EdRAG/resources/materials/practice_lab/edrag_utils.py",
                               "edrag_utils.py")
except Exception as exc:
    print("Could not refresh edrag_utils.py, using the copy already here:", exc)
import edrag_utils
importlib.reload(edrag_utils)
from edrag_utils import *

download({"ITM_Student_Handbook.pdf": "",               # original PDF
          "Tuition_Fees.html": "",                      # original web page
          "ITM_Student_Handbook.md": "converted/"})     # the PDF converted with Docling before class

# The tutorial gateway speaks the OpenAI API, so the standard OpenAI client works with it
llm = OpenAI(base_url=GATEWAY, api_key=HUB_TOKEN)
CHAT_MODEL = "edrag-chat"            # Qwen3.8-Flash on the tutorial server

print("\nGateway:", "OK" if gateway_ok() else "not reachable", "| scored in Notebook 1:", scored_approaches() or "nothing yet")

---
## 1. Convert the documents

A language model reads text. How the PDF and the web page become text decides what the model can see later.

### 1.1 The simple way: extract plain text

In [ ]:
def extract_pdf_plain(path):
    # Join the text layer of every PDF page; layout and tables are lost
    return "\n".join(page.extract_text() or "" for page in PdfReader(path).pages)


def extract_html_plain(path):
    # Keep only the visible text of the web page: menus and footer stay, table structure goes
    return BeautifulSoup(pathlib.Path(path).read_text(encoding="utf-8"), "lxml").get_text("\n")


handbook_plain = extract_pdf_plain(DATA / "ITM_Student_Handbook.pdf")
tuition_plain = extract_html_plain(DATA / "Tuition_Fees.html")
print(f"Handbook: {len(handbook_plain):,} characters | Tuition page: {len(tuition_plain):,} characters\n")
print("--- The undergraduate cost table in plain text ---")
show_region(re.sub(r"\n\s*\n+", "\n", tuition_plain), "Living on Campus", before=60, after=420)

The table has become a column of loose numbers. Which number belongs to which row and which column is now guesswork for the model.

### 1.2 The better way: Markdown that keeps the tables

* **Handbook (PDF):** converted before class with [Docling](https://github.com/docling-project/docling), which recognizes headings and tables. Docling needs a GPU to be fast, so we load its output.
* **Tuition page (HTML):** you convert it now, in three small steps: keep only the main content, spell out merged cells, and convert with `html-to-markdown`.

The undergraduate table has a **merged cell**: in the "Living at home" column, one value of $4,950 covers both Housing and Food.

In [ ]:
guess("step1", "Without any preprocessing, what will the converter put in the Food row of the 'Living at home' column?",
      ["$4,950", "$0", "An empty cell", "$9,900"])

In [ ]:
def main_content(page_html):
    # Keep the page title and the <article> body; navigation, footer, and scripts are dropped
    soup = BeautifulSoup(page_html, "lxml")
    title = soup.select_one("h1").get_text(" ", strip=True)
    return BeautifulSoup(f"<h1>{title}</h1>{soup.select_one('article')}", "lxml")


def html_to_md(soup):
    # html-to-markdown 3.x returns an object; .content holds the Markdown text
    return convert(str(soup)).content


page_html = (DATA / "Tuition_Fees.html").read_text(encoding="utf-8")
tuition_raw_md = html_to_md(main_content(page_html))      # converted without fixing merged cells
for line in food_rows(tuition_raw_md):
    print(line)
GUESS_TRUTH["step1"] = "An empty cell"

The converter left the cell **empty**, which a model can easily read as "food costs $0". Other converters copy $4,950 into both rows instead, so the column no longer adds up to its total. Markdown has no way to write a merged cell, so we fix it ourselves before converting.

> **Core idea:** a converter cannot know what a merged cell means, so we rewrite it ourselves before converting. One line does the work:
> `cell.string = f"{value} (combined: {first} and {others})"`
> When a general-purpose tool gets your data wrong, a few lines based on knowing your own data can fix it.

In [ ]:
def annotate_merged_cells(soup):
    # Find every cell that spans several rows (rowspan), such as the $4,950 shared by Housing and Food
    for cell in soup.select("td[rowspan], th[rowspan]"):
        row = cell.find_parent("tr")
        covered = row.find_next_siblings("tr")[:int(cell["rowspan"]) - 1]   # the rows below that share it
        label = lambda r: " ".join(r.find(["th", "td"]).get_text(" ").split())   # a row's name, e.g. "Food ..."
        value = " ".join(cell.get_text(" ").split())                         # e.g. "$4,950"
        first, others = label(row), ", ".join(label(r) for r in covered)
        col = row.find_all(["th", "td"], recursive=False).index(cell)       # which column it sits in

        # Core idea: write out what the merged value covers
        cell.string = f"{value} (combined: {first} and {others})"
        del cell["rowspan"]

        # Give each covered row its own cell that points back, instead of leaving it empty
        for other in covered:
            filler = soup.new_tag("td")
            filler.string = f"(included in {first})"
            cells = other.find_all(["th", "td"], recursive=False)
            cells[col].insert_before(filler) if col < len(cells) else other.append(filler)
    return soup


tuition_md = html_to_md(annotate_merged_cells(main_content(page_html)))
for line in food_rows(tuition_md):
    print(line)

### 1.3 Clean the handbook's Markdown

Docling's output is much better than plain text but not clean: the table of contents became a table of dot leaders, `&` appears as `&amp;`, image placeholders remain, bullets became symbols from the PDF's private font, and table cells are padded with long runs of spaces.

In [ ]:
def clean_markdown(text):
    lines = []
    for line in text.splitlines():
        if re.search(r"\.{10,}", line):                 # table of contents: rows of dot leaders
            continue
        if line.strip() == "<!-- image -->":             # image placeholders
            continue
        line = html.unescape(line)                       # &amp; -> &
        line = re.sub(r"[\ue000-\uf8ff]", "", line)      # bullet symbols from the PDF's private font
        if line.lstrip().startswith("|"):
            line = re.sub(r" {2,}", " ", line)           # padded table cells
        lines.append(line)
    return re.sub(r"\n{3,}", "\n\n", "\n".join(lines))


handbook_md = clean_markdown((DATA / "ITM_Student_Handbook.md").read_text(encoding="utf-8"))
tuition_md = clean_markdown(tuition_md)
print(f"Handbook Markdown: {len(handbook_md):,} characters | Tuition Markdown: {len(tuition_md):,} characters\n")
check_step1(tuition_md, tuition_raw_md, handbook_md)

---
## 2. Chunk the documents

Retrieval returns pieces of documents, called chunks. A chunk that is too small cuts an answer in half; one that is too large mixes in unrelated text, and a chunk longer than about 512 tokens is cut off by the embedding model.

We compare three presets:

| Preset | Input | Chunking |
| --- | --- | --- |
| A | Plain text | Fixed length: 1,000 characters, 100 overlapping |
| B | Markdown (cleaned) | Fixed length: 1,000 characters, 100 overlapping |
| C | Markdown (cleaned) | By heading, at most 1,500 characters, each chunk starting with its section path |

> **Core idea:** every chunk in preset C starts with its section path, for example
> `Tuition, Fees, and Costs > New Graduate Student (Mies Campus)`.
> A number like $34,650 then always travels with the heading that says what it is. In the code, that is the one line in `section_chunk()`.

In [ ]:
def chunk_fixed(text, source, size=1000, overlap=100):
    # Cut the text every `size` characters; each piece repeats the last `overlap` characters of the one before
    return [{"source": source, "section": "", "text": text[i:i + size]}
            for i in range(0, len(text), size - overlap)]


def section_chunk(path, piece):
    # Core idea: start every chunk with its section path, then a blank line, then the text
    return f"{path}\n\n{piece}"


HANDBOOK, TUITION = "ITM_Student_Handbook.pdf", "Tuition_Fees.html"
PRESETS = {
    "A": chunk_fixed(handbook_plain, HANDBOOK) + chunk_fixed(tuition_plain, TUITION),
    "B": chunk_fixed(handbook_md, HANDBOOK) + chunk_fixed(tuition_md, TUITION),
    # chunk_by_heading() (in edrag_utils.py) splits at each heading and calls section_chunk() for every chunk
    "C": chunk_by_heading(handbook_md, "ITM Graduate Student Handbook", HANDBOOK, section_chunk)
         + chunk_by_heading(tuition_md, "Tuition, Fees, and Costs", TUITION, section_chunk),
}
chunk_stats(PRESETS)

In [ ]:
guess("step2", "If fixed-length chunks are only 300 characters long, will the chunk holding $34,650 still contain the heading 'New Graduate Student'?",
      ["Yes", "No"])

In [ ]:
for name, chunks in PRESETS.items():
    found = chunks_with(chunks, "34,650")
    display(Markdown(f"**Preset {name}**: {len(found)} chunk(s) contain $34,650"))
    for c in found:
        print(c["text"][:300].replace("\n", " ⏎ "), "...\n")

small = chunks_with(chunk_fixed(tuition_md, TUITION, size=300, overlap=30), "34,650")
display(Markdown("**Fixed length, 300 characters**"))
print(small[0]["text"].replace("\n", " ⏎ "))
GUESS_TRUTH["step2"] = "Yes" if any("New Graduate Student" in c["text"] for c in small) else "No"
print("\nDoes the 300-character chunk still name 'New Graduate Student'?", GUESS_TRUTH["step2"])

**Change one setting:** set `SIZE` below to 300, then to 2000, and run the cell each time. How many chunks do you get, and what does the chunk with $34,650 look like?

In [ ]:
SIZE = 1000   # try 300 and 2000

trial = chunk_fixed(handbook_md, HANDBOOK, size=SIZE, overlap=SIZE // 10) + chunk_fixed(tuition_md, TUITION, size=SIZE, overlap=SIZE // 10)
print(f"{len(trial)} chunks of {SIZE} characters\n")
print(chunks_with(trial, "34,650")[0]["text"])

In [ ]:
check_step2(PRESETS)

---
## 3. Embed and retrieve

An embedding model turns each chunk into a list of 384 numbers (a vector), so that texts with similar meaning get similar vectors. We use the small open model **bge-small-en-v1.5**, run by the tutorial server. The vectors for the three presets were computed before class, so this is fast.

In [ ]:
INDEX = {}
for name, chunks in PRESETS.items():
    E, info = embed([c["text"] for c in chunks])      # one row of 384 numbers per chunk
    INDEX[name] = E
    print(f"Preset {name}: matrix {E.shape[0]} x {E.shape[1]} | {info['cached']} from cache | "
          f"{info['truncated']} chunk(s) longer than 512 tokens were cut off")
    if name == "C" and info["truncated"]:
        cut = [c["section"].split(" > ")[-1] for c, n in zip(chunks, info["tokens"]) if n > 512]
        print("   cut off in preset C:", cut)

> **Core idea:** retrieval is one matrix product. All chunk vectors sit in one matrix **E** (one row per chunk) and the question becomes one vector **q**. Because every vector has length 1, `E @ q` scores every chunk against the question at once; sorting the scores picks the top k. That is the one line in `similarity()`.

In [ ]:
def similarity(E, q):
    # Core idea: one score per chunk, the matrix of chunk vectors times the question vector
    return E @ q


def search(question, preset="C", k=5, prefix=True):
    # Embed the question, score every chunk of the preset, and return the k best (score, chunk) pairs
    q, _ = embed([question], is_query=prefix)
    scores = similarity(INDEX[preset], q[0])
    top = np.argsort(-scores)[:k]
    return [(float(scores[i]), PRESETS[preset][i]) for i in top]


show_hits(search(QUESTIONS["Q3"], "C", k=5))

In [ ]:
guess("step3", "If we leave out bge's query prefix, will the top chunk for Q4 change?", ["Yes", "No"])

In [ ]:
with_prefix = search(QUESTIONS["Q4"], "C", k=3, prefix=True)
without = search(QUESTIONS["Q4"], "C", k=3, prefix=False)
print("With the prefix:")
show_hits(with_prefix)
print("\nWithout the prefix:")
show_hits(without)
GUESS_TRUTH["step3"] = "Yes" if with_prefix[0][1] is not without[0][1] else "No"

**Meaning versus keywords.** In the online-course rule, the handbook writes the visa as “Fl Visa” (a lowercase L instead of the digit 1). A keyword search for “F-1” finds other passages that spell it correctly, but misses the rule itself; a vector search can still find it.

In [ ]:
def keyword_search(term, preset="C"):
    # The simplest keyword search: every chunk that contains the term, ignoring upper and lower case
    return [c for c in PRESETS[preset] if term.lower() in c["text"].lower()]


RULE = "one online course may be taken per semester"
hits = keyword_search("F-1")
print(f"Keyword 'F-1' finds {len(hits)} chunk(s):", [c["section"].split(" > ")[-1] for c in hits])
print("  ...including the online-course rule?", any(RULE in c["text"] for c in hits), "\n")
vector_hits = search("How many online courses can F-1 students take?", "C", k=3)
show_hits(vector_hits)
print("\nVector search top 3 includes the rule?", any(RULE in c["text"] for _, c in vector_hits))

**Change one setting:** does each preset retrieve a chunk that holds the answer? The table checks Q1–Q4 by looking for a key phrase of the expected answer among the top **k** chunks. Set `K` to 3, then 8, and run it again.

In [ ]:
K = 5   # try 3 and 8

ANSWER_KEYS = {"Q1": "34,650", "Q2": "53,198", "Q3": "supportdesk@illinoistech.edu", "Q4": "one online course"}
table = {name: {q: "✓" if any(key in c["text"] for _, c in search(QUESTIONS[q], name, k=K)) else "✗"
                for q, key in ANSWER_KEYS.items()} for name in PRESETS}
display(pd.DataFrame(table).T.rename_axis(f"Preset (top {K})"))

In [ ]:
check_step3(similarity, INDEX, PRESETS, search)

---
## 4. Generate the answer

The retrieved chunks go into a prompt, and the language model writes the answer. Here the model is **Qwen3.8-Flash** on the tutorial server. The prompt decides what the model does when the passages do not hold the answer.

Answers are displayed through `show_answer()` (in `edrag_utils.py`). It protects dollar signs, because Jupyter would otherwise treat text between two `$` signs (for example `$34,650 ... $53,198`) as a math formula and swallow it, which makes a correct answer look wrong.

In [ ]:
PROMPT_BASIC = '''Answer the student's question. Here is some information that may help.

{passages}

Question: {question}'''

PROMPT_GROUNDED = '''You answer questions from university students using only the numbered passages below.

Rules:
1. Use only information in the passages. Do not add facts from memory.
2. After each sentence, cite the passages you used, for example [1] or [2][3].
3. If the passages do not contain the answer, say "I could not find this in the provided documents." Then mention any related information the passages do contain.
4. If the answer depends on the student's situation, such as program level or visa status, answer for each case.

{passages}

Question: {question}'''


def answer(question, preset="C", k=5, prompt=PROMPT_GROUNDED, show_prompt=False, quiet=False):
    # 1. Retrieve the k best chunks; 2. number them and put them into the prompt; 3. ask the model.
    # quiet=True returns the answer without printing it (used by the automatic checks).
    hits = search(question, preset, k)
    passages = "\n\n".join(f"[{i}] {c['text']}" for i, (_, c) in enumerate(hits, 1))
    message = prompt.format(passages=passages, question=question)
    reply = llm.chat.completions.create(model=CHAT_MODEL, temperature=0, max_tokens=700,
                                        messages=[{"role": "user", "content": message}])
    text = reply.choices[0].message.content
    if quiet:
        return text
    if show_prompt:
        print(message, "\n" + "=" * 80)
    show_answer(text)                                    # display the answer (dollar signs protected from math rendering)
    for i, (_, c) in enumerate(hits, 1):                 # what each [n] in the answer points to
        print(f"  [{i}] {(c['section'] or c['source'])[:90]}")
    return text

In [ ]:
guess("step4", "With the plain prompt, will the answer to Q2 give one 'international student' tuition figure?", ["Yes", "No"])

In [ ]:
for name, prompt in [("Plain prompt", PROMPT_BASIC), ("Grounded prompt", PROMPT_GROUNDED)]:
    display(Markdown(f"---\n### {name}: Q2"))
    answer(QUESTIONS["Q2"], prompt=prompt)

In [ ]:
for name, prompt in [("Plain prompt", PROMPT_BASIC), ("Grounded prompt", PROMPT_GROUNDED)]:
    display(Markdown(f"---\n### {name}: Q5"))
    answer(QUESTIONS["Q5"], prompt=prompt)

**What to compare.** When retrieval brings back the right passages, a good model may answer well even with the plain prompt. Look instead at what the grounded prompt adds: a citation after each sentence that you can check against the passage list, a fixed sentence when the answer is missing, and answers split by the student's situation. Also check whether either answer adds facts that are not in the passages.

**Change one sentence.** With the grounded prompt, the model says it cannot find an international rate and stops there, without giving the tuition for each program level that the passages do contain. Rule 4 asks for an answer for each case, but the model follows rule 3 first and never gets to rule 4. The next cell rewrites the second half of rule 3 in general terms (not just for Q2): give everything the passages say that could help, separately for each group of students. Then it asks Q2 again. Compare the answer with the grounded answer above.

Want to try your own wording? Edit the new sentence in the cell and run it again. Add `show_prompt=True` to `answer()` to see exactly what the model receives.

In [ ]:
MY_PROMPT = PROMPT_GROUNDED.replace(
    # the sentence in rule 3 we replace ...
    "Then mention any related information the passages do contain.",
    # ... and the new sentence: say what to give, and to split it by group of students
    "Then give everything the passages do say that could help; if it differs for different groups of students, "
    "such as program level or visa status, give it for each group.",
)
answer(QUESTIONS["Q2"], prompt=MY_PROMPT)

**Did the change break anything?** We changed the prompt after looking at one question, so it could make other answers worse. The next cell asks all five questions with both prompts and checks each answer for its key facts automatically (about 20 seconds). A tick is a rough string test, not a full score, but a new cross is a warning sign.

In [ ]:
quick_facts_check(answer, {"Grounded": PROMPT_GROUNDED, "Changed sentence": MY_PROMPT})

In [ ]:
check_step4(answer)

---
## 5. Evaluate your assistant

Run the five questions with your best settings: preset C, top 5, and the improved grounded prompt from step 4 (`MY_PROMPT`).

> **A note on fairness:** we changed the prompt after seeing the answer to Q2, so the score on Q2 is optimistic. In a real project you would tune the prompt on one set of questions (a development set) and report scores on another set that you never used for tuning (a test set). With only five questions we cannot split them, but keep this in mind when you read the scores.

Score each answer with the same rubric as in Notebook 1, using the expected answers on the [test questions page](https://academictutorials.github.io/EdRAG/edrag-test-questions.html):

* **0 = incorrect**: any wrong or invented information, even if the rest is right
* **1 = partially correct**: everything stated is correct, but some expected information is missing
* **2 = completely correct**: correct and complete

In [ ]:
BEST_PROMPT = MY_PROMPT      # the grounded prompt with the sentence improved in step 4


def rag(question, preset="C", k=5, prompt=BEST_PROMPT):
    # The whole pipeline in one call: retrieve, build the prompt, generate
    return answer(question, preset, k, prompt)


for q, text in QUESTIONS.items():
    display(Markdown(f"---\n### {q}: {text}"))
    rag(text)

In [ ]:
# Your overall correctness score for the assistant you built, one number per question
# 0 = incorrect (any wrong or invented information), 1 = partially correct (no errors, something missing), 2 = completely correct
from_scratch_scores_overall = {"Q1": None, "Q2": None, "Q3": None, "Q4": None, "Q5": None}
save_overall("from_scratch", from_scratch_scores_overall)

### Compare all four approaches

In [ ]:
compare_overall()
guess_summary()

---
## 6. Break it (3 minutes)

Write a question that makes your assistant give a wrong or invented answer. Some ideas:

* Ask in another language, for example Chinese or Spanish.
* Ask about something the documents do not cover.
* Ask a question with a false premise: “Since F-1 students may take three online courses per semester, which ones should I choose?”
* Ask about a detail in a table, such as the food cost for an undergraduate living at home.

In [ ]:
rag("Write your question here")

---
## 7. After class (optional)

### 7.1 Hybrid retrieval: keywords plus vectors

BM25 is a classic keyword ranking. Reciprocal rank fusion (RRF) combines two rankings: each chunk gets 1 / (60 + rank) from each list, and the chunks with the highest sums win.

In [ ]:
from rank_bm25 import BM25Okapi

tokenize = lambda t: re.findall(r"\w+", t.lower())             # lower-case words
bm25 = BM25Okapi([tokenize(c["text"]) for c in PRESETS["C"]])   # keyword index over preset C


def hybrid_search(question, k=5, depth=20):
    # Rank with vectors and with keywords, then fuse the two rankings with RRF
    dense = [c for _, c in search(question, "C", k=depth)]
    sparse = [PRESETS["C"][i] for i in np.argsort(-bm25.get_scores(tokenize(question)))[:depth]]
    fused = {}
    for ranking in (dense, sparse):
        for rank, c in enumerate(ranking, 1):
            fused[id(c)] = (fused.get(id(c), (0, c))[0] + 1 / (60 + rank), c)
    return sorted(fused.values(), key=lambda x: -x[0])[:k]


show_hits(hybrid_search("How many online courses can F-1 students take?"))

### 7.2 A vector database

A vector database stores the vectors with their metadata and can filter by metadata. Qdrant runs in memory here, with no server.

In [ ]:
from qdrant_client import QdrantClient, models

qdrant = QdrantClient(":memory:")                               # an in-memory database, gone when the kernel stops
qdrant.create_collection("edrag", vectors_config=models.VectorParams(size=384, distance=models.Distance.COSINE))
qdrant.upsert("edrag", points=[models.PointStruct(id=i, vector=INDEX["C"][i].tolist(), payload=c)
                               for i, c in enumerate(PRESETS["C"])])

q, _ = embed([QUESTIONS["Q1"]], is_query=True)
only_tuition = models.Filter(must=[models.FieldCondition(key="source", match=models.MatchValue(value=TUITION))])
for p in qdrant.query_points("edrag", query=q[0].tolist(), query_filter=only_tuition, limit=3).points:
    print(f"{p.score:.3f}  {p.payload['section']}")

### 7.3 A student profile

Q4's answer depends on visa status. Put what the assistant knows about the student into the question and see whether the answer changes.

In [ ]:
profile = "I am an F-1 international student in my first semester."
rag(f"{profile} {QUESTIONS['Q4']}")

### 7.4 Detailed scoring

Score each criterion separately for the assistant you built, then compare all approaches you scored this way in Notebook 1.

In [ ]:
#                  Relevance  Grounding  Citation  No-unsupported  Refusal
from_scratch_scores = {
    "Q1": [None,      None,      None,     None,           None],
    "Q2": [None,      None,      None,     None,           None],
    "Q3": [None,      None,      None,     None,           None],
    "Q4": [None,      None,      None,     None,           None],
    "Q5": [None,      None,      None,     None,           None],
}
save_detailed("from_scratch", from_scratch_scores)
compare_detailed()

**After the tutorial:** this server will be shut down. The Colab version of this notebook runs the same steps with your own free Gemini API key.